# Walk environment: Gym (CPU), MJX, and mjlab backends

This notebook demonstrates loading the **myoLegWalk** locomotion task on three backends:
1. **Gym (CPU)** — MuJoCo C++ via `gymnasium`; train with SB3 SAC.
2. **MJX** — JAX-based physics; train with **JAX PPO** (Brax) from `scripts/train_sar_jax_ppo.py`.
3. **mjlab** — MuJoCo Warp / Isaac Lab-style API (optional).

For each backend we create the env, train (or run a short rollout), and show rewards; rendering is used where supported.

## Installation (uv)

From the repo root, install all dependencies needed by this notebook:

```bash
uv sync --extra rl --extra mjx --extra mjlab
uv pip install jupyter nbconvert nbclient ipykernel
```

If you use editable installs instead of sync:

```bash
uv pip install -e ".[rl,mjx,mjlab]"
uv pip install jupyter nbconvert nbclient ipykernel
```

In [ ]:
# Ensure repo root on path (for benchmarks.sar_backends adapter)
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "myosuite").exists():
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import myosuite  # noqa: F401 - register envs
from myosuite.utils import gym

TRAIN_STEPS = 100
SEED = 42
SAVE_DIR = Path("./walk_backends_saved")
SAVE_DIR.mkdir(exist_ok=True)

---
## 1. Gym (CPU) backend

Load **myoLegWalk-v0** with `gymnasium`, train 100 steps with SB3 SAC, save the policy, then run a rollout with rendering.

In [ ]:
from stable_baselines3 import SAC

env_gym = gym.make("myoLegWalk-v0", render_mode="rgb_array", seed=SEED)
model_gym = SAC(
    "MlpPolicy",
    env_gym,
    seed=SEED,
    learning_rate=1e-3,
    buffer_size=5000,
    learning_starts=50,
    batch_size=32,
    verbose=0,
    device="cpu",
)
model_gym.learn(total_timesteps=TRAIN_STEPS, log_interval=None)
model_gym.save(SAVE_DIR / "sac_gym_walk")
print(f"Gym: trained {TRAIN_STEPS} steps, policy saved to {SAVE_DIR / 'sac_gym_walk'}")
env_gym.close()

In [ ]:
# Load policy and run rollout with rendering
from IPython.display import HTML, display
from base64 import b64encode

def show_video(frames, width=400):
    if not frames:
        print("No frames to display.")
        return
    try:
        import imageio
        buf = Path(SAVE_DIR / "gym_rollout.mp4")
        imageio.mimsave(buf, frames, fps=25)
        data = buf.read_bytes()
        url = f"data:video/mp4;base64,{b64encode(data).decode()}"
        display(HTML(f'<video width={width} controls><source src="{url}"></video>'))
    except Exception as e:
        print(f"Rendered {len(frames)} frames (save to file or install imageio to show video: {e})")

env_render = gym.make("myoLegWalk-v0", seed=SEED)
model_gym = SAC.load(SAVE_DIR / "sac_gym_walk", env=env_render)
obs, _ = env_render.reset(seed=SEED)
rewards = []
for _ in range(100):
    action, _ = model_gym.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env_render.step(action)
    rewards.append(reward)
    if terminated or truncated:
        break
env_render.close()

# Plot reward over time as a headless-friendly "render"
try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(4, 3))
    plt.plot(rewards)
    plt.xlabel("step")
    plt.ylabel("reward")
    plt.title("Gym rollout reward")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not plot Gym rewards (matplotlib missing?): {e}")

print(f"Gym rollout: {len(rewards)} steps, total reward = {sum(rewards):.3f}")

---
## 2. MJX backend (JAX PPO)

Load **MjxLegWalk-v0** (JAX) and train with **JAX PPO** (Brax PPO on MJX) using the same pipeline as `scripts/train_sar_jax_ppo.py`. Policy is saved under the notebook save dir; we then run a short rollout. MJX has no built-in display renderer—rewards are plotted below (or run the same policy on the Gym env in section 1 to visualize).

In [ ]:
import os

# Use CPU by default for MJX unless JAX_PLATFORMS is explicitly set (e.g. to "cuda").
os.environ.setdefault("JAX_PLATFORMS", "cpu")

# Always run a fast MJX env smoke first.
import jax
import jax.numpy as jp
from myosuite.envs.myo.backends.mjx import make

mjx_env = make("MjxLegWalk-v0")
rng = jax.random.PRNGKey(SEED)
state = mjx_env.reset(rng)
action = jp.zeros((mjx_env.action_size,))
state = mjx_env.step(state, action)
print(f"MJX smoke: reset+step ok, reward={float(state.reward):.4f}")

# Optional JAX PPO train demo (can be expensive on some machines).
RUN_MJX_PPO = os.environ.get("MYOSUITE_RUN_MJX_PPO", "0") == "1"
JAX_PPO_TRAIN_STEPS = 200  # notebook-friendly demo; use 1M+ for real training
JAX_PPO_NUM_ENVS = 8

if RUN_MJX_PPO:
    import wandb
    from scripts.train_sar_jax_ppo import _run_jax_ppo_training

    wandb_run = wandb.init(mode="disabled")  # no login required for demo
    try:
        jax_ppo_result = _run_jax_ppo_training(
            mjx_env_name="MjxLegWalk-v0",
            seed=SEED,
            total_timesteps=JAX_PPO_TRAIN_STEPS,
            num_envs=JAX_PPO_NUM_ENVS,
            output_dir=SAVE_DIR,
            wandb_run=wandb_run,
        )
    except Exception as e:
        jax_ppo_result = None
        print(f"MJX (JAX PPO): failed runtime/setup check: {e}")
        print("If assets are missing, run: git submodule update --init --recursive")
    finally:
        wandb_run.finish()
else:
    jax_ppo_result = None
    print("MJX (JAX PPO): not run by default. Set MYOSUITE_RUN_MJX_PPO=1 to enable.")

if jax_ppo_result is not None:
    make_inference_fn, jax_ppo_params = jax_ppo_result
    print(f"MJX (JAX PPO): trained {JAX_PPO_TRAIN_STEPS} steps, params saved to {SAVE_DIR / 'jax_ppo_params.pkl'}")
else:
    make_inference_fn, jax_ppo_params = None, None

In [ ]:
# Run MJX rollout with trained JAX PPO policy (no display; report rewards)
from scripts.train_sar_jax_ppo import run_jax_ppo_rollout

if make_inference_fn is not None and jax_ppo_params is not None:
    rewards, total = run_jax_ppo_rollout(
        make_inference_fn,
        jax_ppo_params,
        mjx_env_name="MjxLegWalk-v0",
        num_steps=500,
        seed=SEED,
    )
    print(f"MJX rollout: {len(rewards)} steps, total reward = {total:.2f}")
    print("(MJX has no built-in renderer; use the same policy on the Gym env above to visualize.)")
else:
    rewards = []
    print("No JAX PPO policy available; skip rollout.")

In [ ]:
# Visualize MJX rollout as a reward plot (no video, works headless)
import matplotlib.pyplot as plt

if rewards:
    plt.figure(figsize=(4, 3))
    plt.plot(rewards)
    plt.xlabel("step")
    plt.ylabel("reward")
    plt.title("MJX rollout reward")
    plt.tight_layout()
    plt.show()
    print(f"MJX rollout plotted for {len(rewards)} steps.")
else:
    print("No MJX rollout data to plot (run cell 7 first).")

---
## 3. mjlab backend

Load **myoLegWalk-v0** on the mjlab (MuJoCo Warp) backend via `make_env(..., backend="mjlab")`.
Requires `uv sync --extra mjlab` (or `uv pip install -e ".[mjlab]"`).

**Runtime requirement:** use a CUDA-enabled Linux runtime for mjlab rollouts/training.
On macOS or CPU-only runtimes, this section is skipped to avoid known Warp runtime crashes.

**Training note:** mjlab uses **RSL-RL PPO** (on-policy, GPU-native) — not SB3 SAC — because
all physics tensors stay on GPU throughout training. For full training use:

```bash
python benchmarks/sar_backends/run_benchmark.py --backends mjlab
```

The cell below demonstrates the native torch API by running a short rollout when the runtime is supported.

In [ ]:
import platform

try:
    import torch
    from myosuite.core.registry import make_env

    if platform.system() == "Darwin" or not torch.cuda.is_available():
        print(
            "mjlab rollout skipped: requires Linux + CUDA-enabled runtime "
            "(Warp/mjlab can crash on CPU-only or macOS setups)."
        )
    else:
        # Create mjlab backend env (native torch API, no Gymnasium adapter)
        mjlab_env = make_env("myoLegWalk-v0", backend="mjlab")
        mjlab_env.reset(seed=SEED)

        rewards = []
        for _ in range(100):
            # Step directly with torch tensors.
            action = torch.randn(1, mjlab_env.action_space.shape[-1], device=mjlab_env.device)
            obs_dict, rew, term, trunc, info = mjlab_env.step(action)
            rewards.append(float(rew.item() if hasattr(rew, "item") else rew))

            terminated = bool(term.item() if hasattr(term, "item") else term)
            truncated = bool(trunc.item() if hasattr(trunc, "item") else trunc)
            if terminated or truncated:
                break

        mjlab_env.close()
        print(f"mjlab rollout: {len(rewards)} steps, total reward = {sum(rewards):.3f}")

        # Simple reward plot as a proxy for rendering.
        try:
            import matplotlib.pyplot as plt

            plt.figure(figsize=(4, 3))
            plt.plot(rewards)
            plt.xlabel("step")
            plt.ylabel("reward")
            plt.title("mjlab rollout reward (random-action)")
            plt.tight_layout()
            plt.show()
        except Exception as plot_err:
            print(f"Could not plot mjlab rewards (matplotlib missing?): {plot_err}")

        print(
            "For full mjlab PPO training, see "
            "`python benchmarks/sar_backends/run_benchmark.py --backends mjlab`"
        )
except Exception as e:
    print(f"mjlab not available: {e}")
    print("Install with: `uv sync --extra mjlab` or `uv pip install -e \".[mjlab]\"`")